# ImageEval 2026 — Task 1a (Spoken VQA) · **Fanar cascaded API** baseline

An **API-only** alternative to the local-model notebook — **no GPU required**. It runs a
two-stage cascade on the [Fanar API](https://api.fanar.qa/docs):

1. **`Fanar-Aura-STT-1`** transcribes the spoken question + options (`/v1/audio/transcriptions`).
2. **`Fanar-Oryx-IVU-2`** (image understanding) reads the image + transcript and picks the option
   (`/v1/chat/completions` with an `image_url`).

The two stages are **independent functions** (§5) — swap either model, the base URL, or the
whole provider without touching anything else.

> **API key.** Needs a key for **`api.fanar.qa`** (request one at
> [api.fanar.qa/request](https://api.fanar.qa/request)). Paste it when prompted, or set it as a
> Colab secret named `FANAR_API_KEY`. (A QCRI *staging-orchestrator* key will **not** work here —
> that endpoint serves text models only, no STT/vision.)

**Output:** `predictions_<lang>.csv` (`id,raw_prediction,prediction_parsed`) and a Codabench-ready
`prediction_<lang>.zip` — same format as the other baselines. Defaults to `devtest`; set
`SPLIT="dev"` for a local accuracy.

## 1. Install dependencies (light — no torch, no restart)

In [ ]:
!pip install -q -U requests "huggingface_hub[hf_transfer]" pillow tqdm

## 2. Configuration

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

REPO_ID   = "QCRI/ImageEval2026-Task1-AynVQA"
TASK      = "task1a"
LANG      = "en"        # "en" or "msa"  -> output is predictions_<LANG>.csv
SPLIT     = "devtest"   # "devtest"/"test" -> blind (submit) | "dev"/"train" -> labelled (scored)
MAX_RECORDS = 8         # how many records to run; set to None for the whole split

# --- Fanar API (modular: change endpoint / models here) ---
FANAR_BASE_URL = "https://api.fanar.qa/v1"
STT_MODEL = "Fanar-Aura-STT-1"     # long audio? -> "Fanar-Aura-STT-LF-1"
VLM_MODEL = "Fanar-Oryx-IVU-2"     # image-understanding chat model

# Paste your Fanar API key here (request one at api.fanar.qa/request):
FANAR_API_KEY = "YOUR_FANAR_API_KEY"

print(f"config: {TASK}_{LANG} / {SPLIT}  via {FANAR_BASE_URL}")

## 3. Download the split + its media from the Hub

Only the chosen split's JSONL and the exact images/audio it references (cached).

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

jsonl = hf_hub_download(REPO_ID, filename=f"{TASK}/{SPLIT}_{LANG}.jsonl", repo_type="dataset")
records = [json.loads(l) for l in open(jsonl, encoding="utf-8") if l.strip()]
if MAX_RECORDS:
    records = records[:MAX_RECORDS]
print(len(records), "records;  labelled:", "label" in records[0])

def fetch(rel):
    return hf_hub_download(REPO_ID, filename=rel, repo_type="dataset")

needed = sorted({r["image"] for r in records} | {r["audio"] for r in records})
paths = {}
with ThreadPoolExecutor(max_workers=16) as ex:
    for rel, p in tqdm(zip(needed, ex.map(fetch, needed)), total=len(needed), desc="media"):
        paths[rel] = p

## 4. Quick API sanity check

Confirms the key works and the two models are reachable before the full run.

In [ ]:
import requests
HEADERS = {"Authorization": f"Bearer {FANAR_API_KEY}"}
r = requests.get(f"{FANAR_BASE_URL}/models", headers=HEADERS, timeout=30)
r.raise_for_status()
available = {m["id"] for m in r.json().get("data", r.json().get("models", []))}
print("reachable. STT present:", STT_MODEL in available,
      "| vision present:", VLM_MODEL in available)

## 5. The cascade — two independent, swappable stages

`transcribe()` = stage 1 (speech→text). `ask_vlm()` = stage 2 (image+text→answer). Replace
either body to try a different model/provider; the run loop below doesn't change.

In [ ]:
import re, time, base64, mimetypes

def _post(url, *, retries=4, **kw):
    """POST with backoff on transient errors; a 429 (too many requests) stops immediately."""
    for i in range(retries):
        resp = requests.post(url, headers=HEADERS, timeout=120, **kw)
        if resp.status_code == 429:
            raise RuntimeError("API returned 429 (too many requests)")
        if resp.ok:
            return resp
        if i == retries - 1:
            resp.raise_for_status()
        time.sleep(2 ** i)

# ---- stage 1: speech -> text -------------------------------------------
def transcribe(wav_path):
    with open(wav_path, "rb") as f:
        resp = _post(f"{FANAR_BASE_URL}/audio/transcriptions",
                     files={"file": f},
                     data={"model": STT_MODEL, "format": "text"})
    try:
        return resp.json().get("text", resp.text).strip()
    except Exception:
        return resp.text.strip()

# ---- stage 2: image + transcript -> option index -----------------------
PROMPT = ("You are given an image and the transcript of a spoken multiple-choice question "
          "about it. The transcript has the question followed by three answer options in "
          "order (the first option is 0, the second is 1, the third is 2). Using the image, "
          "choose the option that correctly answers the question for THIS image. "
          "Reply with ONLY a single digit: 0, 1, or 2.\n\nTranscript:\n{t}")

def _data_uri(path):
    mime = mimetypes.guess_type(path)[0] or "image/jpeg"
    return f"data:{mime};base64," + base64.b64encode(open(path, "rb").read()).decode()

def ask_vlm(image_path, transcript):
    content = [{"type": "text", "text": PROMPT.format(t=transcript)},
               {"type": "image_url", "image_url": {"url": _data_uri(image_path)}}]
    payload = {"model": VLM_MODEL, "temperature": 0, "max_tokens": 16,
               "messages": [{"role": "user", "content": content}]}
    resp = _post(f"{FANAR_BASE_URL}/chat/completions", json=payload)
    return resp.json()["choices"][0]["message"]["content"].strip()

DEFAULT_INDEX = 0   # fallback when the reply has no 0/1/2 (a blank would be scored wrong)
def parse_index(txt):
    m = re.search(r"[012]", txt)
    return (int(m.group()), True) if m else (DEFAULT_INDEX, False)

def run_one(r):
    raw = ask_vlm(paths[r["image"]], transcribe(paths[r["audio"]]))
    idx, ok = parse_index(raw)
    return r["id"], raw, idx, ok

## 6. Run inference (sequential)

One record at a time (2 API calls each). If an API error occurs, the loop **stops gracefully
and keeps everything done so far**, so the cells below still write and score a partial run.

In [ ]:
rows, n_fallback = [], 0
for r in tqdm(records, desc="infer"):
    try:
        iid, raw, idx, ok = run_one(r)
    except Exception as e:
        print(f"stopped after {len(rows)} items — {e}")
        break
    n_fallback += (not ok)
    rows.append((iid, raw, idx))

print(f"done: {len(rows)} predictions  |  unparseable -> defaulted to {DEFAULT_INDEX}: {n_fallback}")

## 7. Write `predictions_<lang>.csv`

Columns: `id, raw_prediction, prediction_parsed`. Switching `LANG` writes a separate file.

In [ ]:
import csv
OUT_CSV = f"predictions_{LANG}.csv"
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "raw_prediction", "prediction_parsed"])
    for iid, raw, idx in rows:
        w.writerow([iid, raw, idx])
print("wrote", OUT_CSV, ":", len(rows), "rows")

## 8. Score (official metric: **accuracy**)

Runs only when the split is labelled (`dev`/`train`). `devtest`/`test` are blind.

In [ ]:
gold = {r["id"]: r["label"] for r in records if "label" in r}
if gold:
    pred = {iid: idx for iid, _, idx in rows}
    correct = sum(1 for i in gold if pred.get(i) == gold[i])
    print(f"accuracy on {SPLIT}: {correct/len(gold):.4f}  ({correct}/{len(gold)})")
else:
    print(f"'{SPLIT}' is blind (no labels) — submit to Codabench to get the score.")

## 9. Build the Codabench submission

Builds `prediction.csv` (`id,prediction` = the parsed index) and zips it as `prediction_<lang>.zip`.
Submit to [task1a_en](https://www.codabench.org/competitions/17002/) ·
[task1a_msa](https://www.codabench.org/competitions/17001/).

In [ ]:
import zipfile
with open("prediction.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "prediction"])
    for iid, _, idx in rows:
        w.writerow([iid, idx])
zip_name = f"prediction_{LANG}.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("prediction.csv", "prediction.csv")
print("wrote", zip_name, " -> submit this file to Codabench")